# Volume 1. Basic Assembly Calculus

Question: what is the smallest useful assembly-calculus experiment?

This notebook is a visual first contact. You will build a tiny workbench with two stimuli, two source areas, and one target area. Each stimulus forms a sparse constellation of active winners. Then `merge` forms a new assembly in a target area.

The drawings are not anatomy. They are maps of neuron indices, useful for building intuition before moving to more formal overlap and readout diagnostics.

In [ ]:
import math

import matplotlib.pyplot as plt

from neural_assemblies.core.brain import Brain
from neural_assemblies.assembly_calculus import merge, project

The whole experiment has one shape: two inputs form source assemblies, and the two source assemblies jointly drive a target assembly.

In [ ]:
def draw_flow():
    fig, ax = plt.subplots(figsize=(8, 2.8))
    ax.axis("off")

    boxes = {
        "stimulus s1": (0.12, 0.70, "#f2c14e"),
        "area A1": (0.38, 0.70, "#4d9de0"),
        "stimulus s2": (0.12, 0.30, "#f2c14e"),
        "area A2": (0.38, 0.30, "#4d9de0"),
        "merged area B": (0.72, 0.50, "#59a14f"),
    }

    for label, (x, y, color) in boxes.items():
        ax.text(
            x,
            y,
            label,
            ha="center",
            va="center",
            fontsize=12,
            bbox=dict(boxstyle="round,pad=0.35", fc=color, ec="black", lw=1.2),
        )

    arrows = [
        ((0.21, 0.70), (0.31, 0.70)),
        ((0.21, 0.30), (0.31, 0.30)),
        ((0.48, 0.70), (0.62, 0.53)),
        ((0.48, 0.30), (0.62, 0.47)),
    ]
    for start, end in arrows:
        ax.annotate("", xy=end, xytext=start, arrowprops=dict(arrowstyle="->", lw=2))

    ax.text(0.5, 0.08, "project each stimulus, then merge the two source assemblies", ha="center")
    return fig


draw_flow();

Set up the bench. `N` is the number of neurons in each area, `K` is the number of winners, and `BETA` controls Hebbian strengthening after co-activation. The seed keeps the tour repeatable.

In [ ]:
N = 5_000
K = 80
BETA = 0.08
ROUNDS = 8

brain = Brain(p=0.05, save_winners=True, seed=42, engine="numpy_sparse")
brain.add_stimulus("s1", K)
brain.add_stimulus("s2", K)
brain.add_area("A1", n=N, k=K, beta=BETA)
brain.add_area("A2", n=N, k=K, beta=BETA)
brain.add_area("B", n=N, k=K, beta=BETA)

`project` lets a stimulus form a stable assembly in an area. Here `s1` writes into `A1`, and `s2` writes into `A2`.

In [ ]:
a1 = project(brain, "s1", "A1", rounds=ROUNDS)
a2 = project(brain, "s2", "A2", rounds=ROUNDS)

print(f"Source assembly sizes: {len(a1)}, {len(a2)}")
print(f"Source areas: {a1.area}, {a2.area}")

Visualize each assembly as active dots on a square canvas. The dots are winner neuron indices. They are sparse by design: only `K` out of `N` neurons are active.

In [ ]:
def plot_assembly(ax, assembly, n, title, color):
    side = math.ceil(math.sqrt(n))
    winners = assembly.winners
    ax.scatter(winners % side, winners // side, s=14, c=color, alpha=0.80, edgecolors="none")
    ax.set_title(f"{title}\n{len(assembly)} winners")
    ax.set_xlim(-1, side)
    ax.set_ylim(side, -1)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_facecolor("#f7f7f2")


fig, axes = plt.subplots(1, 2, figsize=(8, 4))
plot_assembly(axes[0], a1, N, "A1 from stimulus s1", "#4d9de0")
plot_assembly(axes[1], a2, N, "A2 from stimulus s2", "#e15554")
fig.suptitle("Two source assemblies")
plt.tight_layout()
plt.show()

Checkpoint: a projection gives you a named sparse trace in a named area. You can inspect its size, area, and winners before claiming anything about what it represents.

In [ ]:
merged = merge(brain, "A1", "A2", "B", rounds=5)

print(f"Merged assembly size: {len(merged)}")
print(f"Merged assembly area: {merged.area}")

Now compare the three areas visually. The merged assembly is not the same object as either source assembly; it is a new sparse trace in area `B`, formed by the joint drive from `A1` and `A2`.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.8))
plot_assembly(axes[0], a1, N, "source A1", "#4d9de0")
plot_assembly(axes[1], a2, N, "source A2", "#e15554")
plot_assembly(axes[2], merged, N, "merged target B", "#59a14f")
fig.suptitle("Sparse assemblies as inspectable traces")
plt.tight_layout()
plt.show()

Try next: change `K` from `80` to `40` or `120`, rerun the notebook, and watch the constellations become sparser or denser. Then move to the readout notebook, where overlap becomes the main diagnostic for comparing assemblies in the same area.